# Load and align neural and saved DINO features

This notebook does only three things:

1. load the neural raster;
2. load one already-extracted DINO feature matrix;
3. reorder the DINO image axis to match the neural image axis.

No model is loaded and no new features are extracted.

In [ ]:
import os
import sys
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import yaml
from torchvision.datasets import ImageFolder

# Locate the repository whether Jupyter starts at the root or in scripts/.
cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in [cwd, *cwd.parents] if (path / "config.yaml").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the project config.yaml.")
# end if PROJECT_ROOT is None

ENV = os.getenv("MY_ENV", "tiziano_mac_mini")
with open(PROJECT_ROOT / "config.yaml", "r") as f:
    config = yaml.safe_load(f)
# end with open config
paths = config[ENV]["paths"]
sys.path.append(paths["src_path"])
sys.path.append(paths["useful_stuff_path"])

from project_specific_utils.dataloader import (  # noqa: E402
    load_img_natraster,
    map_image_order_from_ann_to_monkey,
)

In [ ]:
@dataclass
class Cfg:
    # Neural data.
    monkey_name: str = "three0"
    date: str = "250313"
    brain_area: str = "AIT"
    new_fs: int = 100

    # Saved DINO features.
    folder_name: str = "talia_20each_tizi"
    model_name: str = "dino_v3_l"
    img_size: int = 224
    layer_name: str = "layer.13.mlp.down_proj"
    pooling: str = "mean"
    feature_subdir: str = "models"


cfg = Cfg()
cfg

## 1. Load the neural raster

The project loader returns a `TimeSeries` whose array is ordered as `[neurons, time, images]`.

In [ ]:
raster = load_img_natraster(
    paths=paths,
    monkey_name=cfg.monkey_name,
    date=cfg.date,
    new_fs=cfg.new_fs,
    brain_area=cfg.brain_area,
)
neural_features = raster.get_array()

if neural_features.ndim != 3:
    raise ValueError(
        f"Expected neural [neurons, time, images], got {neural_features.shape}."
    )
# end if neural features have unexpected axes

print(f"neural [neurons, time, images]: {neural_features.shape}")
print(f"sampling frequency: {raster.fs} Hz")

## 2. Load the saved DINO features

Each saved layer is an `.npz` file containing `arr_0` with shape `[features, images]`. In the current data tree the raw activations are under `models/`; `results/` contains derived dRSA outputs.

In [ ]:
feature_name = (
    f"{cfg.folder_name}_{cfg.model_name}_{cfg.img_size}_{cfg.layer_name}"
    f"_features_{cfg.pooling}pool.npz"
)
feature_path = Path(paths["data_path"]) / cfg.feature_subdir / feature_name

if not feature_path.is_file():
    raise FileNotFoundError(feature_path)
# end if feature file is missing

with np.load(feature_path) as feature_file:
    dino_features = feature_file["arr_0"]
# end with feature file

if dino_features.ndim != 2:
    raise ValueError(
        f"Expected DINO [features, images], got {dino_features.shape}."
    )
# end if DINO features have unexpected axes

print(f"DINO before alignment [features, images]: {dino_features.shape}")
print(f"loaded from: {feature_path}")

## 3. Align the image axes

The saved DINO features follow `ImageFolder` order. `idx_ord[j]` gives the DINO image index corresponding to neural image index `j`, so it is applied only to the DINO image axis. `ImageFolder` is used here only to recover filenames in their original extraction order; it does not load image pixels.

In [ ]:
stimuli_root = Path(paths["livingstone_lab"]) / "Stimuli" / cfg.folder_name
image_dataset = ImageFolder(root=stimuli_root, allow_empty=True)

idx_ord = np.asarray(
    map_image_order_from_ann_to_monkey(
        paths, cfg.monkey_name, cfg.date, image_dataset
    ),
    dtype=int,
)
dino_features_aligned = dino_features[:, idx_ord]

if neural_features.shape[2] != dino_features_aligned.shape[1]:
    raise ValueError(
        "Neural and DINO image counts differ after alignment: "
        f"{neural_features.shape[2]} and {dino_features_aligned.shape[1]}."
    )
# end if aligned image counts differ

print(f"alignment indices: {idx_ord.shape}")
print(f"neural [neurons, time, images]: {neural_features.shape}")
print(f"DINO aligned [features, images]: {dino_features_aligned.shape}")

## Result

`neural_features[:, :, image_idx]` and `dino_features_aligned[:, image_idx]` now refer to the same stimulus. These two arrays are ready for a time-specific neural slice, RSA, or a linear mapping.